In [1]:
source = "I am a boy."
target = "Ich bin ein Junge."

# Encoder-Decoder RNN
# Encoder["I am a boy"] -> hidden state (context vector)
# Next Token Prediction 
# 1. Decoder[hidden state, "<bos>"] -> "ich"
# 2. Decoder[hidden state, ("<bos>", "ich")] -> "bin"
# 3. Decoder[hidden state, ("<bos>", "ich", "bin")] -> "ein"
# 4. Decoder[hidden state, ("<bos>", "ich", "bin", "ein")] -> "Junge"
# 5. Decoder[hidden state, ("<bos>", "ich", "bin", "ein", "Junge")] -> "."
# 6. Decoder[hidden state, ("<bos>", "ich", "bin", "ein", "Junge", ".")] -> "<eos>"

# X = (hidden state, ("<bos>", "ich", "bin", "ein", "Junge", "."))
# y = ("ich", "bin", "ein", "Junge", ".", "<eos>")

In [3]:
import re, json, torch, torch.nn as nn
from torch.utils.data import DataLoader

path = "./deu.txt"

lines = open(path, encoding="utf-8").read().strip().split("\n")
lines = lines[:20000] # 

pairs = [ln.split("\t")[:2] for ln in lines] 
src_texts, tgt_texts = zip(*pairs)

In [ ]:
PAD, UNK, BOS, EOS = 0, 1, 2, 3

VOCAB_SIZE = 20004

def tokenize(s): return re.findall(r"\b\w+\b", s.lower())
def build_vocab(texts, max_tokens=VOCAB_SIZE):
    from collections import Counter
    freq = Counter(tok for t in texts for tok in tokenize(t))
    itos = ["<pad>", "<unk>", "<bos>", "<eos>"] + [w for w,_ in freq.most_common(max_tokens-4)]
    return {w:i for i,w in enumerate(itos)}, itos
src_texts_vocab, src_itos = build_vocab(src_texts)
tgt_texts_vocab, tgt_itos = build_vocab(tgt_texts)
src_texts_vocab

In [ ]:
def vectorize(text, stoi, max_len, add_bos_eos=False):
    ids = [stoi.get(tok, UNK) for tok in tokenize(text)]
    if add_bos_eos: ids = [BOS] + ids + [EOS]
    ids = ids[:max_len]
    if len(ids) < max_len: ids += [PAD]*(max_len-len(ids))
    return ids

max_src, max_tgt = 30, 30 

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src = torch.tensor([vectorize(t, src_texts_vocab, max_src) for t in src_batch])
    tgt = torch.tensor([vectorize(t, tgt_texts_vocab, max_tgt, add_bos_eos=True) for t in tgt_batch])
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    return src, tgt_in, tgt_out

dataset = list(zip(src_texts, tgt_texts))
loader = DataLoader(dataset, batch_size= 64, shuffle=True, collate_fn=collate_fn)



In [17]:
sentence = "this is sample sentence for embedding"
dc = {s:i for i,s in enumerate(sorted(sentence.replace(',', '').split()))}
dc

{'embedding': 0, 'for': 1, 'is': 2, 'sample': 3, 'sentence': 4, 'this': 5}

In [19]:
vocab_size_tmp = len(dc)
emb = torch.nn.Embedding(vocab_size_tmp, 3)
emb.weight.data 

tensor([[-1.0553,  0.8355,  0.8856],
        [ 0.4593, -0.3672, -0.6491],
        [-0.1048,  0.7617, -1.0307],
        [-0.9383, -1.1477,  1.9689],
        [-1.2533,  0.4653, -1.0407],
        [ 0.8341, -1.1649,  1.4309]])

In [ ]:
emb_dim = 128 # In Praxis startet man mit 768 
hid_dim = 256 

class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD) 
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first = True)
    
    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x) # feature extractor
        return hidden 

class Decoder(nn.Module):
    def __init__(self, vocab_size): 
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD) 
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first = True) # feature extractor 
        self.fc = nn.Linear(hid_dim, vocab_size) # classifier head, MLP-head, next-token 
    
    def forward(self,x, h): # h stammt aus Encoder!
        x = self.embedding(x)
        out,_ = self.rnn(x,h) 
        return self.fc(out) # Entscheidung, was ist der nächste Token 

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec 

    def forward(self, src, tgt_in): 
        # src ... englische Sätze
        # tgt_in ... bereits ins Deutsch übersetze Teilsätze 
        hidden_enc = self.enc(src)
        logits = self.dec(tgt_in, hidden_enc)
        return logits

device = "mps"

model = Seq2Seq(
    Encoder(len(src_texts_vocab)), # Englisch
    Decoder(len(tgt_texts_vocab)), # Deutsch
).to(device)
        


In [26]:
crit = nn.CrossEntropyLoss(ignore_index=PAD) # Indizes wo PAD-Token fließen nicht in Gradientberechnung ein  
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
epochs = 20 


@torch.no_grad()
def translate(prompt, max_len=max_tgt):
    model.eval()
    src = torch.tensor([vectorize(prompt, src_texts_vocab, max_src)], device=device)
    h = model.enc(src)
    ys = torch.tensor([[BOS]], device=device)
    out_tokens = []
    for _ in range(max_len):
        logits = model.dec(ys, h)
        next_id = logits[0, -1].argmax().item()
        if next_id in (EOS, PAD): break
        out_tokens.append(next_id)
        ys = torch.cat([ys, torch.tensor([[next_id]], device=device)], dim=1)
    return " ".join(tgt_itos[t] for t in out_tokens)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 
    for src, tgt_in, tgt_out in loader:
        src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
        logits = model(src,tgt_in)
        loss = crit(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1)) 
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # gradient clipping ->  preventing exploding gradient 
        optimizer.step()
        running_loss += loss.item()
    print(f"epoch {epoch+1}: loss {running_loss/len(loader):.4f}") # 
    print(translate("I will do my best."))

epoch 1: loss 4.4353
ich habe einen
epoch 2: loss 3.1975
ich habe mich gesehen
epoch 3: loss 2.5819
ich werde mich beeilen
epoch 4: loss 2.1571
ich werde das leben
epoch 5: loss 1.8292
ich werde mein bestes tun
epoch 6: loss 1.5594
ich werde mein bestes holen
epoch 7: loss 1.3301
ich werde mich verstecken
epoch 8: loss 1.1348
ich werde mein bestes tun


KeyboardInterrupt: 